# Pond Inlet -- Sentinel-1 Patch Extraction (Cross-Region Training + Test Data)

Serves two purposes, per Michel's suggestion to (1) test the existing
Tuktoyaktuk-trained model on a genuinely new region, and (2) do "model
development" -- train a new model in one region and test it in the
other:

1. **Inference test**: match Sentinel-1 imagery to Pond Inlet's
   already-provided LiDAR patches (11,197 patches, verified genuinely
   Pond Inlet via CRS reprojection -- 8.0km from the townsite), for
   running the existing `09`/`10` Tuktoyaktuk-trained checkpoint on
   (no retraining).
2. **New training data**: the same extracted patches double as the
   training set for a from-scratch Pond-Inlet-trained model in a
   follow-up notebook, with its own spatial-block train/val split.

## Coverage note -- why the date selection differs from `02`/`11`

Unlike Tuktoyaktuk and Cambridge Bay, Pond Inlet's Sentinel-1 RTC
coverage on Planetary Computer is concentrated almost entirely in
January-March each year, with essentially no April coverage in *any*
year across the archive (checked directly: a full year/month histogram
across 98 total scenes for this AOI shows zero April entries). A
`SEARCH_DAYS`-windowed search around the LiDAR survey date
(`2024-04-26`) found nothing usable, so this notebook instead selects
the three scenes closest to the survey date from an **unrestricted**
search of the whole archive:

| Scene date | Days from survey date |
|---|---|
| 2023-03-17 | -405.1 |
| 2023-03-05 | -417.1 |
| 2023-02-21 | -429.1 |

**This is a real, documented limitation**: all three are ~13.5 months
before the survey, not a close date match. The redeeming factor is that
February-March is the same frozen/snow-covered season as the April
survey -- a far smaller seasonal mismatch than Cambridge Bay's only
option (May 2025, clearly post-thaw). Still, this is not a clean
same-year match and should be reported as such.

## Setup

In [1]:
import os
import json
import datetime as dt
from datetime import timezone
import glob as glob_module
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import rasterio
from rasterio.warp import transform_bounds, transform_geom
from rasterio.windows import Window, from_bounds
from shapely.geometry import box, shape
from shapely.ops import unary_union

import pystac_client
import planetary_computer
from dotenv import load_dotenv

load_dotenv()
if os.environ.get('PC_SDK_SUBSCRIPTION_KEY'):
    planetary_computer.settings.set_subscription_key(os.environ['PC_SDK_SUBSCRIPTION_KEY'])

## Configuration

In [2]:
REPO_DIR = Path('/cs/student/project_msc/2025/aibh/jiayiche')
INPUT_DIR = REPO_DIR / 'input_data'
REGION = 'pondinlet'
LIDAR_DIR = INPUT_DIR / 'lidar_patches_pondinlet_extracted' / 'lidar_patches_pondinlet'
SURVEY_DATE = dt.date(2024, 4, 26)
N_SCENES = 3  # matches CONTEXT_K used throughout this project

PATCH_SIZE = 256
S1_PATCH_SIZE = int(round(PATCH_SIZE / 10))  # 26 px @ 10m

MERGED_DIR = REPO_DIR / 'raw_data' / f'{REGION}_pc_rtc_merged'
OUT_S1_DIR = INPUT_DIR / f's1_patches_{REGION}_pcrtc'
MERGED_DIR.mkdir(parents=True, exist_ok=True)
OUT_S1_DIR.mkdir(parents=True, exist_ok=True)
print('LIDAR_DIR:', LIDAR_DIR)
print('MERGED_DIR:', MERGED_DIR)
print('OUT_S1_DIR:', OUT_S1_DIR)

LIDAR_DIR: /cs/student/project_msc/2025/aibh/jiayiche/input_data/lidar_patches_pondinlet_extracted/lidar_patches_pondinlet
MERGED_DIR: /cs/student/project_msc/2025/aibh/jiayiche/raw_data/pondinlet_pc_rtc_merged
OUT_S1_DIR: /cs/student/project_msc/2025/aibh/jiayiche/input_data/s1_patches_pondinlet_pcrtc


## 1. Build the AOI and select the nearest available scenes

Same AOI-building function as `02`/`11`. Unlike those notebooks, this
does an **unrestricted** search (no date window) and manually picks the
`N_SCENES` closest to `SURVEY_DATE`, since Pond Inlet's coverage pattern
is too sparse/seasonally skewed for a fixed-window search to find
anything.

In [3]:
def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if not paths:
        raise FileNotFoundError(f'No LiDAR patches found in {patches_dir}')
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]

    def read_bounds(path):
        with rasterio.open(path) as src:
            return src.crs, src.bounds

    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(read_bounds, p) for p in paths]
        for future in as_completed(futures):
            results.append(future.result())
    crs = results[0][0]
    native = unary_union([box(*bounds) for _, bounds in results])
    geojson = transform_geom(crs, 'EPSG:4326', native.__geo_interface__)
    return shape(geojson).buffer(0)

aoi = aoi_from_lidar_patches(LIDAR_DIR)
aoi_ll = aoi.convex_hull
print('AOI bounds (lon, lat):', aoi_ll.bounds)

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
search = catalog.search(collections=['sentinel-1-rtc'], intersects=aoi_ll.__geo_interface__)
all_items = list(search.items())
print(f'Total scenes for this AOI (all time): {len(all_items)}')

target = dt.datetime.combine(SURVEY_DATE, dt.time(), tzinfo=timezone.utc)
items = sorted(all_items, key=lambda it: abs((it.datetime - target).total_seconds()))[:N_SCENES]
items = sorted(items, key=lambda it: it.datetime)  # chronological order for t0/t1/t2 naming

print(f'\nSelected {len(items)} nearest scenes to {SURVEY_DATE.isoformat()}:')
for i, item in enumerate(items):
    days_off = (item.datetime - target).total_seconds() / 86400
    print(f'  t{i}: {item.id} | {item.datetime} | {days_off:+.1f} days from survey')

AOI bounds (lon, lat): (-78.62837866403368, 72.69983676650693, -77.6792464187407, 72.85140815833994)
Total scenes for this AOI (all time): 98

Selected 3 nearest scenes to 2024-04-26:
  t0: S1A_IW_GRDH_1SSH_20230221T221912_20230221T221937_047340_05AEA9_rtc | 2023-02-21 22:19:24.507218+00:00 | -429.1 days from survey
  t1: S1A_IW_GRDH_1SSH_20230305T221912_20230305T221937_047515_05B493_rtc | 2023-03-05 22:19:24.990268+00:00 | -417.1 days from survey
  t2: S1A_IW_GRDH_1SSH_20230317T221912_20230317T221937_047690_05BA7F_rtc | 2023-03-17 22:19:24.739356+00:00 | -405.1 days from survey


## 2. Merge VV+VH into local 2-band GeoTIFFs, windowed to the AOI

Identical windowed-HTTPS-read approach as `02`/`11`.

In [4]:
merged_paths = []
merged_attrs = []

for i, item in enumerate(items):
    with rasterio.open(item.assets['vv'].href) as vv_src:
        aoi_bounds_scene_crs = transform_bounds('EPSG:4326', vv_src.crs, *aoi_ll.bounds)
        window = from_bounds(*aoi_bounds_scene_crs, transform=vv_src.transform)
        vv = vv_src.read(1, window=window)
        out_transform = rasterio.windows.transform(window, vv_src.transform)
        out_crs = vv_src.crs
    with rasterio.open(item.assets['vh'].href) as vh_src:
        vh_window = from_bounds(*transform_bounds('EPSG:4326', vh_src.crs, *aoi_ll.bounds), transform=vh_src.transform)
        vh = vh_src.read(1, window=vh_window)

    h = min(vv.shape[0], vh.shape[0])
    w = min(vv.shape[1], vh.shape[1])
    stacked = np.stack([vv[:h, :w], vh[:h, :w]]).astype(np.float32)

    out_path = MERGED_DIR / f't{i}.tif'
    meta = {'driver': 'GTiff', 'count': 2, 'height': h, 'width': w,
            'dtype': 'float32', 'crs': out_crs, 'transform': out_transform}
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(stacked)
    merged_paths.append(str(out_path))

    props = item.properties
    merged_attrs.append({
        'acquisition_date': item.datetime.date().isoformat(),
        'orbit_direction': 'ASCENDING' if props.get('sat:orbit_state') == 'ascending' else 'DESCENDING',
        'relative_orbit_number': props.get('sat:relative_orbit'),
    })
    print(f'Wrote t{i}.tif: shape={stacked.shape}, nodata_frac={float((stacked == -32768.0).mean()):.4f}')

attrs_json_path = MERGED_DIR / 'attrs.json'
with open(attrs_json_path, 'w') as jf:
    json.dump(merged_attrs, jf, indent=2)
print('Wrote', attrs_json_path)

KeyError: 'vv'

In [5]:
from collections import Counter
pol_counts = Counter(item.properties.get('sat:polarizations', tuple(item.assets.keys())) for item in all_items)
print(pol_counts)

# filter to only dual-pol (VV+VH) scenes
dualpol_items = [it for it in all_items if 'vv' in it.assets and 'vh' in it.assets]
print(f'\n{len(dualpol_items)} / {len(all_items)} scenes have both VV and VH')

target = dt.datetime.combine(SURVEY_DATE, dt.time(), tzinfo=timezone.utc)
nearest_dualpol = sorted(dualpol_items, key=lambda it: abs((it.datetime - target).total_seconds()))[:5]
for it in nearest_dualpol:
    days_off = (it.datetime - target).total_seconds() / 86400
    print(f'{it.id} | {it.datetime} | {days_off:+.1f} days')


Counter({('hh', 'tilejson', 'rendered_preview'): 90, ('hh', 'hv', 'tilejson', 'rendered_preview'): 8})

0 / 98 scenes have both VV and VH


In [6]:
hhhv_items = [it for it in all_items if 'hh' in it.assets and 'hv' in it.assets]
print(f'{len(hhhv_items)} HH+HV scenes:')
for it in sorted(hhhv_items, key=lambda it: it.datetime):
    days_off = (it.datetime - target).total_seconds() / 86400
    print(f'{it.id} | {it.datetime} | {days_off:+.1f} days from survey')


8 HH+HV scenes:
S1A_IW_GRDH_1SDH_20150514T120210_20150514T120239_005917_0079FA_rtc | 2015-05-14 12:02:25.173653+00:00 | -3269.5 days from survey
S1A_IW_GRDH_1SDH_20160829T121034_20160829T121059_012815_014355_rtc | 2016-08-29 12:10:46.704493+00:00 | -2796.5 days from survey
S1A_IW_GRDH_1SDH_20160919T114614_20160919T114639_013121_014D5B_rtc | 2016-09-19 11:46:27.080001+00:00 | -2775.5 days from survey
S1A_IW_GRDH_1SDH_20170304T120238_20170304T120303_015542_0198A9_rtc | 2017-03-04 12:02:50.655968+00:00 | -2609.5 days from survey
S1A_IW_GRDH_1SDH_20170306T114610_20170306T114635_015571_01998B_rtc | 2017-03-06 11:46:23.345223+00:00 | -2607.5 days from survey
S1A_IW_GRDH_1SDH_20170309T121031_20170309T121056_015615_019AD2_rtc | 2017-03-09 12:10:43.880415+00:00 | -2604.5 days from survey
S1A_IW_GRDH_1SDH_20190704T120242_20190704T120307_027967_03286C_rtc | 2019-07-04 12:02:55.280276+00:00 | -1757.5 days from survey
S1A_IW_GRDH_1SDH_20190926T120300_20190926T120325_029192_0350B4_rtc | 2019-09-26 1

**Check before continuing**: confirm `nodata_frac` is near 0 for every
`t{i}.tif` above.

## 3. Match against Pond Inlet's existing LiDAR patches

Same matching functions as `02`/`11`, copied verbatim.

In [7]:
def build_s1_products_from_corrected(geotiff_paths, attrs_jsons=None):
    products = []
    for i, path in enumerate(geotiff_paths):
        src = rasterio.open(path)
        attrs = attrs_jsons[i] if attrs_jsons and i < len(attrs_jsons) else None
        products.append({"src": src, "crs": src.crs, "transform": src.transform,
                          "height": src.height, "width": src.width, "attrs": attrs})
    if not products:
        raise ValueError("No Sentinel-1 products loaded -- check geotiff_paths.")
    return products


def close_products(products):
    for p in products:
        p["src"].close()


def extract_lidar_matched_s1_patches(lidar_patches_dir, sentinel1_products, s1_patch_size,
                                      out_s1_dir, pattern="lidar_patch_*.tif", max_nan_frac=0.02):
    lidar_paths = sorted(glob_module.glob(os.path.join(str(lidar_patches_dir), pattern)))
    print(f"Found {len(lidar_paths)} existing LiDAR patches to match against "
          f"{len(sentinel1_products)} Sentinel-1 product(s).")

    n_written, n_skipped, n_skipped_nan = 0, 0, 0

    for idx, lp in enumerate(lidar_paths):
        if idx % 500 == 0:
            print(f"  ...processed {idx}/{len(lidar_paths)} "
                  f"(written: {n_written}, skipped: {n_skipped}, skipped-NaN: {n_skipped_nan})")

        patch_id = os.path.splitext(os.path.basename(lp))[0].split("_")[-1]

        with rasterio.open(lp) as lsrc:
            lidar_bounds = lsrc.bounds
            lidar_crs = lsrc.crs

        s1_patches, s1_transforms = [], []
        ok = True
        has_too_much_nan = False
        for prod in sentinel1_products:
            try:
                s1_bounds = transform_bounds(lidar_crs, prod["crs"], *lidar_bounds, densify_pts=21)
                window = from_bounds(*s1_bounds, transform=prod["transform"]).round_offsets().round_lengths()
                r0, c0 = int(window.row_off), int(window.col_off)
                hh, ww = int(window.height), int(window.width)

                if (hh, ww) != (s1_patch_size, s1_patch_size):
                    ok = False; break
                if r0 < 0 or c0 < 0:
                    ok = False; break
                if r0 + s1_patch_size > prod["height"] or c0 + s1_patch_size > prod["width"]:
                    ok = False; break

                read_window = Window(c0, r0, s1_patch_size, s1_patch_size)
                patch = prod["src"].read(window=read_window)
                if patch.shape[1:] != (s1_patch_size, s1_patch_size):
                    ok = False; break

                nan_frac = float(np.mean(np.isnan(patch)))
                if nan_frac > max_nan_frac:
                    has_too_much_nan = True
                    break

                s1_patches.append(patch)
                s1_transforms.append(rasterio.windows.transform(read_window, prod["transform"]))
            except Exception:
                ok = False
                break

        if has_too_much_nan:
            n_skipped_nan += 1
            continue

        if not ok or len(s1_patches) != len(sentinel1_products):
            n_skipped += 1
            continue

        patch_dir = os.path.join(str(out_s1_dir), f"s1_patch_{patch_id}")
        os.makedirs(patch_dir, exist_ok=True)

        attrs_list = []
        for ti, (prod, patch, tr) in enumerate(zip(sentinel1_products, s1_patches, s1_transforms)):
            meta = {
                "driver": "GTiff", "count": patch.shape[0],
                "height": s1_patch_size, "width": s1_patch_size,
                "dtype": "float32", "crs": prod["crs"], "transform": tr,
            }
            with rasterio.open(os.path.join(patch_dir, f"t{ti}.tif"), "w", **meta) as dst:
                dst.write(patch.astype(np.float32))
            attrs_list.append(prod.get("attrs"))

        with open(os.path.join(patch_dir, "attrs.json"), "w") as jf:
            json.dump(attrs_list, jf, indent=2)

        n_written += 1

    print(f"Done. Matched: {n_written}, skipped (out of bounds/wrong size): {n_skipped}, "
          f"skipped (too much NaN, >{max_nan_frac:.0%}): {n_skipped_nan}")
    return n_written, n_skipped

In [8]:
products_pcrtc = build_s1_products_from_corrected(merged_paths, attrs_jsons=merged_attrs)
extract_lidar_matched_s1_patches(LIDAR_DIR, products_pcrtc, S1_PATCH_SIZE, OUT_S1_DIR)
close_products(products_pcrtc)

ValueError: No Sentinel-1 products loaded -- check geotiff_paths.

## 4. Verify the output

In [ ]:
sample_patches = sorted(glob_module.glob(os.path.join(str(OUT_S1_DIR), 's1_patch_*')))
lidar_total = len(glob_module.glob(os.path.join(str(LIDAR_DIR), 'lidar_patch_*.tif')))
print(f'Total patches written: {len(sample_patches)} (out of {lidar_total} Pond Inlet LiDAR patches)')

if sample_patches:
    with rasterio.open(os.path.join(sample_patches[0], 't0.tif')) as src:
        print('Sample patch shape:', src.shape, f'(expect {S1_PATCH_SIZE}x{S1_PATCH_SIZE})')
    for f in sorted(glob_module.glob(os.path.join(sample_patches[0], 't*.tif'))):
        with rasterio.open(f) as src:
            arr = src.read()
            print(f'  {os.path.basename(f)}: finite_frac={np.isfinite(arr).mean():.4f}, '
                  f'nonzero_frac={(arr != 0).mean():.4f}, min/max={arr.min():.5f}/{arr.max():.5f}')

from collections import Counter
counts = Counter(len(glob_module.glob(os.path.join(p, 't*.tif'))) for p in sample_patches)
print('Distribution of timesteps per patch:', dict(sorted(counts.items())))

## Next steps (separate notebooks)

1. **Inference test**: load `09`'s (or `10`'s) Tuktoyaktuk-trained
   checkpoint, run it on these Pond Inlet patches with no retraining and
   no train/val split (every patch held out), compute the same
   reconstruction metrics for direct comparison against Tuktoyaktuk's
   in-region numbers.
2. **New training run**: use these same patches as a training set --
   with their own spatial-block train/val split (mirroring `09`/`10`'s
   leakage-safe approach) -- to train a from-scratch Pond-Inlet model,
   then optionally test it back on Tuktoyaktuk for the full two-way
   cross-region comparison.